# Demo: Limpieza y análisis de datos con **Python + Pandas**
**Objetivo didáctico:** mostrar un flujo completo (carga → diagnóstico → limpieza → validación → análisis básico) y evidenciar el *antes* y *después* del tratamiento de datos.

> Este cuaderno usa un dataset **sintético** (generado aquí mismo) con errores comunes: valores faltantes, duplicados, tipos inconsistentes, fechas mal formateadas, textos con espacios/casos distintos, etc.


In [ ]:
# 1) Librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## 2) Dataset de ejemplo (con problemas intencionales)

In [ ]:
# Creamos un dataset con problemas típicos de calidad
raw_data = [
    {"FacturaID": "F-0001", "Fecha": "2026-02-01", "Cliente": "  ana perez ", "Ciudad": "guatemala", "Categoria": "Bebidas", "Producto": "Café", "Cantidad": "2", "PrecioUnitario": "Q 18.50", "Descuento": None},
    {"FacturaID": "F-0002", "Fecha": "01/02/2026", "Cliente": "Juan  López", "Ciudad": "Guatemala ", "Categoria": "BEBIDAS", "Producto": "Te", "Cantidad": "1", "PrecioUnitario": "Q18.50", "Descuento": "Q 0.00"},
    {"FacturaID": "F-0003", "Fecha": "2026/02/02", "Cliente": "maria-garcia", "Ciudad": "Mixco", "Categoria": "Snacks", "Producto": "Galletas", "Cantidad": "3", "PrecioUnitario": "Q 7,25", "Descuento": "Q 1.00"},
    {"FacturaID": "F-0003", "Fecha": "2026/02/02", "Cliente": "maria-garcia", "Ciudad": "Mixco", "Categoria": "Snacks", "Producto": "Galletas", "Cantidad": "3", "PrecioUnitario": "Q 7,25", "Descuento": "Q 1.00"},  # duplicado
    {"FacturaID": "F-0004", "Fecha": "2026-02-03", "Cliente": None, "Ciudad": "Antigua", "Categoria": "Snacks", "Producto": "Papas", "Cantidad": "-2", "PrecioUnitario": "Q 12.00", "Descuento": "Q 0.00"},  # cantidad negativa
    {"FacturaID": "F-0005", "Fecha": "2026-02-03", "Cliente": "Luis", "Ciudad": None, "Categoria": "Lácteos", "Producto": "Leche", "Cantidad": "2", "PrecioUnitario": "Q 9.5", "Descuento": "Q 0.50"},
    {"FacturaID": "F-0006", "Fecha": "2026-02-04", "Cliente": "Sofía", "Ciudad": "Villa Nueva", "Categoria": "Lácteos", "Producto": "Yogurt", "Cantidad": "1000", "PrecioUnitario": "Q 6.00", "Descuento": "Q 0.00"},  # outlier cantidad
    {"FacturaID": "F-0007", "Fecha": "2026-02-04", "Cliente": "Carlos", "Ciudad": "Mixco", "Categoria": "Bebidas", "Producto": "Agua", "Cantidad": "2", "PrecioUnitario": None, "Descuento": "Q 0.00"},  # precio faltante
]

df_raw = pd.DataFrame(raw_data)

# Guardamos una copia "antes" de limpiar
df_before = df_raw.copy()

df_before


## 3) Diagnóstico rápido de calidad

In [ ]:
df_before.info()


In [ ]:
# Conteo de valores faltantes por columna
df_before.isna().sum().sort_values(ascending=False)


In [ ]:
# Duplicados
df_before.duplicated().sum()


## 4) Limpieza / tratamiento de datos

### Reglas de limpieza (para clase)
- Normalizar textos: recortar espacios, estandarizar mayúsculas/minúsculas.
- Convertir `Fecha` a tipo fecha.
- Convertir `Cantidad` a numérico y corregir valores negativos.
- Convertir `PrecioUnitario` y `Descuento` a numérico (quitando `Q`, separadores, etc.).
- Manejar faltantes (imputación simple).
- Eliminar duplicados.
- Tratar outliers (ejemplo: recorte por percentiles).


In [ ]:
df = df_before.copy()

# --- Textos: trim + normalización ---
def norm_text(s):
    if pd.isna(s):
        return s
    # quitar espacios dobles y extremos
    s = re.sub(r"\s+", " ", str(s)).strip()
    return s

import re

df["Cliente"] = df["Cliente"].apply(norm_text).str.title()
df["Ciudad"] = df["Ciudad"].apply(norm_text).str.title()
df["Categoria"] = df["Categoria"].apply(norm_text).str.title()
df["Producto"] = df["Producto"].apply(norm_text).str.title()

# --- Fechas: parseo robusto ---
df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce", dayfirst=True)

# --- Cantidad: numérico ---
df["Cantidad"] = pd.to_numeric(df["Cantidad"], errors="coerce")

# Corregir cantidades negativas (regla simple para demo: valor absoluto)
df.loc[df["Cantidad"] < 0, "Cantidad"] = df.loc[df["Cantidad"] < 0, "Cantidad"].abs()

# --- Moneda: Q 18.50 / Q 7,25 / Q18.50 ---
def money_to_float(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    s = s.replace("Q", "").replace(" ", "")
    s = s.replace(",", ".")  # manejo de coma decimal
    # eliminar cualquier cosa que no sea número, punto o signo
    s = re.sub(r"[^0-9\.-]", "", s)
    return pd.to_numeric(s, errors="coerce")

df["PrecioUnitario"] = df["PrecioUnitario"].apply(money_to_float)
df["Descuento"] = df["Descuento"].apply(money_to_float)

# --- Faltantes: imputación simple ---
# Cliente faltante: etiqueta
df["Cliente"] = df["Cliente"].fillna("Desconocido")

# Ciudad faltante: etiqueta
df["Ciudad"] = df["Ciudad"].fillna("Desconocida")

# Precio faltante: imputar con mediana por categoría (fallback a mediana global)
median_global = df["PrecioUnitario"].median()
df["PrecioUnitario"] = df.groupby("Categoria")["PrecioUnitario"].transform(lambda s: s.fillna(s.median()))
df["PrecioUnitario"] = df["PrecioUnitario"].fillna(median_global)

# Descuento faltante: 0
df["Descuento"] = df["Descuento"].fillna(0)

# --- Duplicados ---
df = df.drop_duplicates()

# --- Outliers: recorte por percentiles en Cantidad (demo) ---
p01, p99 = df["Cantidad"].quantile([0.01, 0.99])
df["Cantidad"] = df["Cantidad"].clip(lower=p01, upper=p99)

# Resultado final
df_after = df.copy()
df_after


## 5) Evidencia: *Antes* vs *Después*

La idea es que el estudiante vea explícitamente:
- cambios de tipos (strings → fechas/números)
- reducción de faltantes
- eliminación de duplicados
- estandarización de textos


In [ ]:
print("ANTES (primeras filas):")
display(df_before.head())

print("\nDESPUÉS (primeras filas):")
display(df_after.head())


In [ ]:
# Comparación de calidad: faltantes por columna
resumen_calidad = pd.DataFrame({
    "faltantes_antes": df_before.isna().sum(),
    "faltantes_despues": df_after.isna().sum(),
    "dtype_antes": df_before.dtypes.astype(str),
    "dtype_despues": df_after.dtypes.astype(str)
})
resumen_calidad


## 6) Análisis básico (ejemplo)

Calculamos una métrica de negocio simple: **TotalNeto = Cantidad * PrecioUnitario - Descuento**.
Luego hacemos un resumen por ciudad y categoría.


In [ ]:
df_after["TotalNeto"] = df_after["Cantidad"] * df_after["PrecioUnitario"] - df_after["Descuento"]

df_after[["FacturaID", "Fecha", "Cliente", "Ciudad", "Categoria", "Producto", "Cantidad", "PrecioUnitario", "Descuento", "TotalNeto"]]


In [ ]:
# Resumen por Ciudad
res_ciudad = df_after.groupby("Ciudad", as_index=False)["TotalNeto"].sum().sort_values("TotalNeto", ascending=False)
res_ciudad


In [ ]:
# Gráfico rápido
plt.figure(figsize=(8,4))
plt.bar(res_ciudad["Ciudad"], res_ciudad["TotalNeto"])
plt.xticks(rotation=30, ha="right")
plt.ylabel("Total Neto (Q)")
plt.title("Ventas netas por ciudad (demo)")
plt.tight_layout()
plt.show()


In [ ]:
# Resumen por Categoría
res_categoria = df_after.groupby("Categoria", as_index=False)["TotalNeto"].sum().sort_values("TotalNeto", ascending=False)
res_categoria


## 7) Exportación (opcional)

En una tarea real, normalmente se exporta el dataset limpio a CSV/Parquet para su uso posterior.


In [ ]:
output_path = "datos_limpios_demo.csv"
df_after.to_csv(output_path, index=False, encoding="utf-8")
print("Archivo exportado:", output_path)


## 8) Actividades sugeridas para el alumnado
1. Cambiar la regla de corrección de cantidades negativas (¿eliminar registros vs. valor absoluto?).
2. Probar otra estrategia de imputación para `PrecioUnitario` (promedio por producto, KNN, etc.).
3. Crear una función `quality_report(df)` que devuelva:
   - dtypes
   - faltantes
   - duplicados
   - valores únicos por columna
4. Agregar validaciones (asserts) al final, por ejemplo:
   - `PrecioUnitario > 0`
   - `Cantidad >= 0`
   - `Fecha` no nula
